# Bronze Layer — MovieLens Raw Ingest

**Purpose:** Copy raw MovieLens tables from `workspace.default` into `workspace.bronze` as Delta tables with zero transformations.

Databricks uploaded the CSVs and created the source tables automatically in `workspace.default`.
This notebook promotes them into the Bronze schema, which is the official start of our Medallion pipeline.

| Step | Action |
|------|--------|
| 1 | Create `workspace.bronze` schema |
| 2 | Read all 5 tables from `workspace.default` |
| 3 | Save as Bronze Delta tables in `workspace.bronze` |
| 4 | Confirm row counts |

## Config

Two schema paths are all we need — no file paths or credentials required.
Databricks handles all access through Unity Catalog.

In [ ]:
# Source: where Databricks auto-created the tables when we uploaded the CSVs
SOURCE_SCHEMA = "workspace.default"

# Target: our Bronze layer — all raw tables live here
TARGET_SCHEMA = "workspace.bronze"

# All 5 source tables we want to promote to Bronze
TABLES = ["ratings", "movies", "links", "tags", "genome_tags"]

## Step 1 — Create Bronze schema

A schema (also called a database) is the namespace that groups our Bronze tables together.
`IF NOT EXISTS` makes this safe to re-run — it does nothing if the schema already exists.

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_SCHEMA}")

print(f"Schema ready: {TARGET_SCHEMA}")

## Step 2 — Read from workspace.default

`spark.table()` reads a Unity Catalog table by its full three-part name: `catalog.schema.table`.
This replaces `spark.read.csv()` — no file paths or DBFS needed.

In [ ]:
# Read all 5 source tables into DataFrames
ratings_df     = spark.table(f"{SOURCE_SCHEMA}.ratings")
movies_df      = spark.table(f"{SOURCE_SCHEMA}.movies")
links_df       = spark.table(f"{SOURCE_SCHEMA}.links")
tags_df        = spark.table(f"{SOURCE_SCHEMA}.tags")
genome_tags_df = spark.table(f"{SOURCE_SCHEMA}.genome_tags")

# Quick sanity check — print each schema so we can verify columns and types
for name, df in [("ratings", ratings_df), ("movies", movies_df), ("links", links_df),
                 ("tags", tags_df), ("genome_tags", genome_tags_df)]:
    print(f"--- {name} ---")
    df.printSchema()

## Step 3 — Save as Bronze Delta tables

Each table is written to `workspace.bronze` with the `bronze_` prefix so it is immediately
clear which layer a table belongs to when browsing the catalog.

`mode("overwrite")` makes the notebook **idempotent** — safe to re-run without duplicating data.
`overwriteSchema` allows the column types to change between runs during development.

In [ ]:
# Map each DataFrame to its Bronze target table name
tables_to_write = {
    "bronze_ratings":     ratings_df,
    "bronze_movies":      movies_df,
    "bronze_links":       links_df,
    "bronze_tags":        tags_df,
    "bronze_genome_tags": genome_tags_df,
}

for table_name, df in tables_to_write.items():
    target = f"{TARGET_SCHEMA}.{table_name}"
    print(f"Writing {target} ...")

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target)
    )

    print(f"  Done.")

print("\nAll Bronze tables written.")

## Step 4 — Confirm row counts

We read back from the Bronze Delta tables (not the DataFrames in memory) to confirm
the data actually landed on disk correctly.

In [ ]:
# Expected counts from our earlier data exploration
expected = {
    "bronze_ratings":     25_000_095,
    "bronze_movies":          62_423,
    "bronze_links":           62_423,
    "bronze_tags":         1_093_360,
    "bronze_genome_tags":      1_128,
}

print("=" * 55)
print("  Bronze ingest complete")
print("=" * 55)
print(f"  {'Table':<22} {'Rows':>12}  {'Match?'}")
print("-" * 55)

for table_name, exp_count in expected.items():
    actual = spark.table(f"{TARGET_SCHEMA}.{table_name}").count()
    match  = "OK" if actual == exp_count else f"MISMATCH (expected {exp_count:,})"
    print(f"  {table_name:<22} {actual:>12,}  {match}")

print("=" * 55)